In [13]:
from dl import authClient
authClient.login('sbowes')

'No password or token supplied'

In [8]:
import pandas as pd
import numpy as np
from dl import queryClient
from astropy.table import Table, vstack

# ------------------------------
# User settings
# ------------------------------
input_file = 'summary_results15.csv'   # file with RA/DEC
output_file = 'smash_targets.fits'     # Combined output FITS
ra_col = 'RA'                           # RA column name
dec_col = 'DEC'                         # DEC column name
search_radius_deg = 0.0002778           # 1 arcsec radius
chunk_size = 200                        # Number of points per query chunk

# ------------------------------
# Load target table
# ------------------------------
targets = pd.read_csv(input_file)
# Check column names
print(targets.columns)

print(f"Loaded {len(targets)} targets from {input_file}")

# ------------------------------
# Split into chunks to avoid overly long queries
# ------------------------------
raw_chunks = np.array_split(targets, max(1, len(targets) // chunk_size + 1))
chunks = [pd.DataFrame(c, columns=targets.columns) for c in raw_chunks]  # Force columns

all_results = []
chunk_summary = []

for i, chunk in enumerate(chunks, 1):
    print(f"\nProcessing chunk {i}/{len(chunks)} with {len(chunk)} targets")
    
    # Build ADQL q3c query
    clauses = [
        f"q3c_radial_query(ra, dec, {row[ra_col]}, {row[dec_col]}, {search_radius_deg})"
        for _, row in chunk.iterrows()
    ]
    
    adql_query = f"""
    SELECT *
    FROM smash_dr2.object
    WHERE {" OR ".join(clauses)}
    """
    
    # Run query and save results
    result_file = f'chunk_{i}.fits'
    print("Submitting query...")
    queryClient.query(sql=adql_query, fmt='fits', out=result_file)
    
    # Check if any rows were returned
    t = Table.read(result_file)
    n_rows = len(t)
    chunk_summary.append((i, n_rows))
    
    if n_rows > 0:
        all_results.append(result_file)
        print(f"Chunk {i} returned {n_rows} rows and saved to {result_file}")
    else:
        print(f"Chunk {i} returned 0 rows, skipping")

# ------------------------------
# Combine non-empty FITS files
# ------------------------------
if all_results:
    tables = [Table.read(f) for f in all_results]
    combined_table = vstack(tables)
    combined_table.write(output_file, overwrite=True)
    print(f"\nAll chunks combined into {output_file}")
else:
    print("\nNo data returned from any chunk!")

# ------------------------------
# Summary
# ------------------------------
print("\nChunk summary (chunk number, rows returned):")
for i, n in chunk_summary:
    print(f"Chunk {i}: {n} rows")



Index(['RA', 'DEC', 'overall_mean', 'overall_median', 'overall_mean_mag',
       'overall_median_mag', 'offset_flag', 'chi_squared_68',
       'chi2_threshold_68', 'chi_flag_68', 'chi_squared_95',
       'chi2_threshold_95', 'chi_flag_95', 'chi_squared_997',
       'chi2_threshold_997', 'chi_flag_997', 'amplitude_5',
       'lower_percentile_5', 'upper_percentile_5', 'mag_amplitude_5',
       'amplitude_1', 'lower_percentile_1', 'upper_percentile_1',
       'mag_amplitude_1', 'largest_amp', 'best_period', 'alarm_level_flag',
       'std_amplitude', 'logT', 'logL', 'cephied_like', 'ir_variable_like',
       'other_variable_like'],
      dtype='str')
Loaded 848 targets from summary_results15.csv

Processing chunk 1/5 with 170 targets
Submitting query...
Chunk 1 returned 223 rows and saved to chunk_1.fits

Processing chunk 2/5 with 170 targets
Submitting query...
Chunk 2 returned 213 rows and saved to chunk_2.fits

Processing chunk 3/5 with 170 targets
Submitting query...
Chunk 3 returned

In [12]:
# read in and print coordinates of successfully matched sources
def read_matched_coords(fits_file):
    table = Table.read(fits_file)
    coords = np.array([(row['ra'], row['dec']) for row in table])
    print(f"Read {len(coords)} matched coordinates from {fits_file}")
    return coords
read_matched_coords(output_file)

for coord in read_matched_coords(output_file):
    print(f"RA: {coord[0]}, DEC: {coord[1]}")
    # find closest source in original list
    dists = np.sqrt((targets['RA'] - coord[0])**2 + (targets['DEC'] - coord[1])**2)
    closest_idx = np.argmin(dists)
    print(f"Closest target index: {closest_idx}, RA: {targets.iloc[closest_idx]['RA']}, DEC: {targets.iloc[closest_idx]['DEC']}")
    # store so we can check if all matched sources are found
    matched_target_indices.append(closest_idx)
    # break after first for brevity
    break



Read 1011 matched coordinates from smash_targets.fits
Read 1011 matched coordinates from smash_targets.fits
RA: 9.3801432121013, DEC: -74.41106480068613
Closest target index: 13, RA: 9.380191, DEC: -74.411057
RA: 8.763891657175002, DEC: -74.20340727360475
Closest target index: 10, RA: 8.763859, DEC: -74.203384
RA: 7.521817569740818, DEC: -73.91388363389902
Closest target index: 0, RA: 7.521702, DEC: -73.913918
RA: 9.123991771135369, DEC: -73.91863568933518
Closest target index: 12, RA: 9.124077, DEC: -73.918602
RA: 7.746501877050463, DEC: -73.74141393508198
Closest target index: 1, RA: 7.746443, DEC: -73.741425
RA: 8.22883175639301, DEC: -73.82196793923505
Closest target index: 6, RA: 8.228736, DEC: -73.821953
RA: 8.637638961822047, DEC: -73.87335847163156
Closest target index: 8, RA: 8.637665, DEC: -73.873337
RA: 8.98919879035233, DEC: -73.73407374004566
Closest target index: 11, RA: 8.989162, DEC: -73.734062
RA: 7.948915729072618, DEC: -73.59924514155145
Closest target index: 3, RA: 

In [10]:
# find which objects there are multiple of

from collections import Counter
def find_duplicates(fits_file):
    table = Table.read(fits_file)
    coord_list = [(row['ra'], row['dec']) for row in table]
    coord_counts = Counter(coord_list)
    duplicates = {coord: count for coord, count in coord_counts.items() if count > 1}
    print(f"Found {len(duplicates)} duplicate coordinates in {fits_file}")
    for coord, count in duplicates.items():
        print(f"Coordinate {coord} appears {count} times")
    return duplicates
find_duplicates(output_file)

Found 0 duplicate coordinates in smash_targets.fits


{}

# Analysis of SMASH DR2 Query Results

Let's analyze the results from our SMASH DR2 query to understand:
1. Which target coordinates have multiple matches
2. Which target coordinates are missing from the results
3. Statistics on the matching process

In [19]:
import pandas as pd
import numpy as np
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units as u
from collections import defaultdict

# Load the original targets and results
targets = pd.read_csv(input_file)
smash_results = Table.read(output_file)

print(f"Original targets: {len(targets)}")
print(f"SMASH DR2 results: {len(smash_results)}")
print(f"Expected 848, got {len(smash_results)} results")
print(f"Extra matches: {len(smash_results) - len(targets)}")

Original targets: 848
SMASH DR2 results: 1011
Expected 848, got 1011 results
Extra matches: 163


In [20]:
# Debug: Check data types and sample values
print("\nTargets data info:")
print(f"RA dtype: {targets['RA'].dtype}")
print(f"DEC dtype: {targets['DEC'].dtype}")
print(f"Sample RA values: {targets['RA'].head().values}")
print(f"Sample DEC values: {targets['DEC'].head().values}")
print(f"Any NaN in RA: {targets['RA'].isna().any()}")
print(f"Any NaN in DEC: {targets['DEC'].isna().any()}")

print("\nSMASH results data info:")
print(f"ra dtype: {smash_results['ra'].dtype}")
print(f"dec dtype: {smash_results['dec'].dtype}")
print(f"Sample ra values: {smash_results['ra'][:5]}")
print(f"Sample dec values: {smash_results['dec'][:5]}")


Targets data info:
RA dtype: float64
DEC dtype: float64
Sample RA values: [7.521702 7.746443 7.925977 7.948884 7.980047]
Sample DEC values: [-73.913918 -73.741425 -73.53167  -73.599243 -73.578545]
Any NaN in RA: False
Any NaN in DEC: False

SMASH results data info:
ra dtype: >f8
dec dtype: >f8
Sample ra values:         ra       
-----------------
  9.3801432121013
8.763891657175002
7.521817569740818
9.123991771135369
7.746501877050463
Sample dec values:        dec        
------------------
-74.41106480068613
-74.20340727360475
-73.91388363389902
-73.91863568933518
-73.74141393508198


In [21]:
# Create SkyCoord objects for efficient matching
# Convert to numpy arrays and ensure they're float64
target_ra_vals = np.array(targets['RA'], dtype=np.float64)
target_dec_vals = np.array(targets['DEC'], dtype=np.float64)
smash_ra_vals = np.array(smash_results['ra'], dtype=np.float64)
smash_dec_vals = np.array(smash_results['dec'], dtype=np.float64)

target_coords = SkyCoord(ra=target_ra_vals*u.deg, dec=target_dec_vals*u.deg)
smash_coords = SkyCoord(ra=smash_ra_vals*u.deg, dec=smash_dec_vals*u.deg)

# For each SMASH result, find the closest target coordinate
idx_closest_targets, d2d, d3d = smash_coords.match_to_catalog_sky(target_coords)

# Convert distances to arcseconds
distances_arcsec = d2d.to(u.arcsec).value

print(f"Distance statistics (arcsec):")
print(f"  Min: {distances_arcsec.min():.4f}")
print(f"  Max: {distances_arcsec.max():.4f}")
print(f"  Mean: {distances_arcsec.mean():.4f}")
print(f"  Median: {np.median(distances_arcsec):.4f}")

# Check if any matches exceed our search radius (1 arcsec)
exceeds_radius = distances_arcsec > 1.0
n_exceeds = np.sum(exceeds_radius)
print(f"\nMatches exceeding 1 arcsec radius: {n_exceeds}")
if n_exceeds > 0:
    print(f"  Max distance for these: {distances_arcsec[exceeds_radius].max():.4f} arcsec")
    print("  (This shouldn't happen with q3c_radial_query!)")

Distance statistics (arcsec):
  Min: 0.0021
  Max: 1.0000
  Mean: 0.2298
  Median: 0.1141

Matches exceeding 1 arcsec radius: 0


In [22]:
# Analyze multiple matches for each target coordinate
target_match_counts = defaultdict(list)
for i, target_idx in enumerate(idx_closest_targets):
    target_match_counts[target_idx].append(i)

# Create summary statistics
match_counts = [len(matches) for matches in target_match_counts.values()]
unique_targets_matched = len(target_match_counts)
targets_with_multiple_matches = sum(1 for count in match_counts if count > 1)

print(f"\nMatching Summary:")
print(f"  Total target coordinates: {len(targets)}")
print(f"  Targets with at least one match: {unique_targets_matched}")
print(f"  Targets with multiple matches: {targets_with_multiple_matches}")
print(f"  Targets with no matches: {len(targets) - unique_targets_matched}")
print(f"  Total SMASH objects found: {len(smash_results)}")

print(f"\nMatch count distribution:")
from collections import Counter
count_distribution = Counter(match_counts)
for n_matches in sorted(count_distribution.keys()):
    n_targets = count_distribution[n_matches]
    print(f"  {n_targets} targets have {n_matches} match(es)")


Matching Summary:
  Total target coordinates: 848
  Targets with at least one match: 816
  Targets with multiple matches: 154
  Targets with no matches: 32
  Total SMASH objects found: 1011

Match count distribution:
  662 targets have 1 match(es)
  122 targets have 2 match(es)
  23 targets have 3 match(es)
  9 targets have 4 match(es)


In [23]:
# Detailed analysis of targets with multiple matches
print("=== TARGETS WITH MULTIPLE MATCHES ===")
multiple_match_details = []

for target_idx, smash_indices in target_match_counts.items():
    if len(smash_indices) > 1:
        target_ra = targets.iloc[target_idx]['RA']
        target_dec = targets.iloc[target_idx]['DEC']
        
        print(f"\nTarget {target_idx}: RA={target_ra:.6f}, DEC={target_dec:.6f}")
        print(f"  Found {len(smash_indices)} matches:")
        
        match_info = {
            'target_idx': target_idx,
            'target_ra': target_ra,
            'target_dec': target_dec,
            'n_matches': len(smash_indices),
            'smash_indices': smash_indices,
            'match_details': []
        }
        
        for i, smash_idx in enumerate(smash_indices):
            smash_ra = smash_results['ra'][smash_idx]
            smash_dec = smash_results['dec'][smash_idx]
            distance = distances_arcsec[smash_idx]
            
            print(f"    Match {i+1}: SMASH idx={smash_idx}, RA={smash_ra:.6f}, DEC={smash_dec:.6f}, dist={distance:.3f}\"")
            
            match_details = {
                'smash_idx': smash_idx,
                'smash_ra': smash_ra,
                'smash_dec': smash_dec,
                'distance_arcsec': distance
            }
            match_info['match_details'].append(match_details)
            
        multiple_match_details.append(match_info)

print(f"\nFound {len(multiple_match_details)} targets with multiple matches")

=== TARGETS WITH MULTIPLE MATCHES ===

Target 5: RA=7.981742, DEC=-73.585571
  Found 4 matches:
    Match 1: SMASH idx=9, RA=7.981183, DEC=-73.585696, dist=0.725"
    Match 2: SMASH idx=10, RA=7.981785, DEC=-73.585581, dist=0.057"
    Match 3: SMASH idx=11, RA=7.981216, DEC=-73.585481, dist=0.626"
    Match 4: SMASH idx=12, RA=7.982138, DEC=-73.585431, dist=0.644"

Target 4: RA=7.980047, DEC=-73.578545
  Found 4 matches:
    Match 1: SMASH idx=13, RA=7.979233, DEC=-73.578518, dist=0.834"
    Match 2: SMASH idx=14, RA=7.980073, DEC=-73.578656, dist=0.400"
    Match 3: SMASH idx=15, RA=7.980068, DEC=-73.578550, dist=0.027"
    Match 4: SMASH idx=16, RA=7.980707, DEC=-73.578618, dist=0.722"

Target 2: RA=7.925977, DEC=-73.531670
  Found 2 matches:
    Match 1: SMASH idx=17, RA=7.926168, DEC=-73.531815, dist=0.557"
    Match 2: SMASH idx=18, RA=7.925979, DEC=-73.531678, dist=0.028"

Target 43: RA=11.938212, DEC=-73.307259
  Found 2 matches:
    Match 1: SMASH idx=39, RA=11.938252, DEC=-73.

In [24]:
# Find missing targets (those with no matches)
matched_target_indices = set(target_match_counts.keys())
all_target_indices = set(range(len(targets)))
missing_target_indices = all_target_indices - matched_target_indices

print("=== MISSING TARGETS (NO MATCHES FOUND) ===")
print(f"Number of missing targets: {len(missing_target_indices)}")

if len(missing_target_indices) > 0:
    missing_details = []
    for target_idx in sorted(missing_target_indices):
        target_ra = targets.iloc[target_idx]['RA']
        target_dec = targets.iloc[target_idx]['DEC']
        
        missing_details.append({
            'target_idx': target_idx,
            'ra': target_ra,
            'dec': target_dec
        })
        
        print(f"  Target {target_idx}: RA={target_ra:.6f}, DEC={target_dec:.6f}")
    
    # Save missing targets to file for further investigation
    missing_df = pd.DataFrame(missing_details)
    missing_df.to_csv('missing_smash_targets.csv', index=False)
    print(f"\nMissing targets saved to 'missing_smash_targets.csv'")
else:
    print("Great! All targets have at least one match.")

=== MISSING TARGETS (NO MATCHES FOUND) ===
Number of missing targets: 32
  Target 378: RA=69.807742, DEC=-71.735428
  Target 391: RA=72.301680, DEC=-69.456535
  Target 411: RA=73.033181, DEC=-66.818192
  Target 413: RA=73.073257, DEC=-66.912819
  Target 415: RA=73.222376, DEC=-67.095154
  Target 425: RA=73.566326, DEC=-66.301208
  Target 426: RA=73.573663, DEC=-67.093658
  Target 433: RA=73.738997, DEC=-66.752464
  Target 438: RA=73.839980, DEC=-67.436485
  Target 445: RA=74.063274, DEC=-66.203690
  Target 446: RA=74.073144, DEC=-66.305267
  Target 451: RA=74.127341, DEC=-66.302498
  Target 470: RA=74.437683, DEC=-65.708374
  Target 500: RA=75.247440, DEC=-66.644096
  Target 509: RA=75.744805, DEC=-65.937920
  Target 518: RA=76.041510, DEC=-67.330482
  Target 529: RA=76.336289, DEC=-70.741852
  Target 533: RA=76.474828, DEC=-68.180702
  Target 562: RA=77.286327, DEC=-68.985405
  Target 587: RA=78.378220, DEC=-69.539902
  Target 592: RA=78.507998, DEC=-67.451942
  Target 593: RA=78.5184

In [25]:
# Create a comprehensive mapping table
print("=== CREATING COMPREHENSIVE MAPPING TABLE ===")

# Create a table that maps each original target to its SMASH matches
mapping_data = []

for target_idx in range(len(targets)):
    target_ra = targets.iloc[target_idx]['RA']
    target_dec = targets.iloc[target_idx]['DEC']
    
    if target_idx in target_match_counts:
        # Has matches
        smash_indices = target_match_counts[target_idx]
        n_matches = len(smash_indices)
        
        for i, smash_idx in enumerate(smash_indices):
            smash_ra = smash_results['ra'][smash_idx]
            smash_dec = smash_results['dec'][smash_idx]
            distance = distances_arcsec[smash_idx]
            
            mapping_data.append({
                'target_idx': target_idx,
                'target_ra': target_ra,
                'target_dec': target_dec,
                'n_matches_total': n_matches,
                'match_number': i + 1,
                'smash_idx': smash_idx,
                'smash_ra': smash_ra,
                'smash_dec': smash_dec,
                'distance_arcsec': distance,
                'has_match': True
            })
    else:
        # No matches
        mapping_data.append({
            'target_idx': target_idx,
            'target_ra': target_ra,
            'target_dec': target_dec,
            'n_matches_total': 0,
            'match_number': 0,
            'smash_idx': -1,
            'smash_ra': np.nan,
            'smash_dec': np.nan,
            'distance_arcsec': np.nan,
            'has_match': False
        })

# Convert to DataFrame and save
mapping_df = pd.DataFrame(mapping_data)
mapping_df.to_csv('target_smash_mapping.csv', index=False)

print(f"Created comprehensive mapping table with {len(mapping_df)} rows")
print(f"Saved to 'target_smash_mapping.csv'")
print(f"Columns: {list(mapping_df.columns)}")

# Quick summary of the mapping
print(f"\nMapping summary:")
print(f"  Rows with matches: {mapping_df['has_match'].sum()}")
print(f"  Rows without matches: {(~mapping_df['has_match']).sum()}")
print(f"  Unique targets: {mapping_df['target_idx'].nunique()}")
print(f"  Total SMASH objects: {mapping_df['smash_idx'][mapping_df['smash_idx'] >= 0].nunique()}")

=== CREATING COMPREHENSIVE MAPPING TABLE ===
Created comprehensive mapping table with 1043 rows
Saved to 'target_smash_mapping.csv'
Columns: ['target_idx', 'target_ra', 'target_dec', 'n_matches_total', 'match_number', 'smash_idx', 'smash_ra', 'smash_dec', 'distance_arcsec', 'has_match']

Mapping summary:
  Rows with matches: 1011
  Rows without matches: 32
  Unique targets: 848
  Total SMASH objects: 1011


## Comments on Your SMASH DR2 Download Approach

**What you did well:**

1. **Smart chunking**: Breaking the queries into chunks of 200 targets is excellent - it avoids timeout issues and makes the process more manageable.

2. **Using q3c_radial_query**: This is the correct spatial indexing function for SMASH DR2. It's much more efficient than manual distance calculations.

3. **1 arcsec search radius**: This is a reasonable choice - tight enough to avoid too many false matches but large enough to account for small astrometric differences between catalogs.

4. **Saving intermediate results**: Saving each chunk as a separate FITS file is smart for debugging and recovery if something goes wrong.

**Potential improvements:**

1. **Consider using ADQL DISTANCE function**: Instead of multiple OR clauses, you could use a single query with the DISTANCE function, though your approach works fine.

2. **Add error handling**: Consider adding try/except blocks around the query calls in case of network issues.

3. **Consider coordinate frame**: Make sure your target coordinates are in the same frame as SMASH (should be ICRS/J2000).

**The multiple matches are expected** because:
- Some of your targets might be near bright stars that have been observed multiple times
- SMASH has some duplicate entries in crowded fields
- The 1 arcsec radius might catch genuine nearby sources in dense stellar fields

The analysis above will help you understand exactly what's happening with your matches!

# Summary of SMASH DR2 Query Results

## Key Findings:

**Overall Match Statistics:**
- **Original targets**: 848
- **SMASH objects found**: 1,011 
- **Targets with matches**: 816 (96.2%)
- **Missing targets**: 32 (3.8%)
- **Multiple matches**: 154 targets have more than one match

**Multiple Match Distribution:**
- 662 targets have exactly 1 match
- 122 targets have exactly 2 matches  
- 23 targets have exactly 3 matches
- 9 targets have exactly 4 matches

**Distance Analysis:**
- All matches are within the 1 arcsec search radius ✓
- Distance range: 0.002" to 1.000"
- Median distance: 0.11" (tight matches!)
- Mean distance: 0.23"

## Files Created:
1. **`target_smash_mapping.csv`** - Complete mapping table (1,043 rows)
2. **`missing_smash_targets.csv`** - 32 targets without matches

## Interpretation:

The **extra 163 matches** (1011 - 848) come from:
- **154 targets with multiple SMASH matches** (explains most extra matches)
- This is expected in dense stellar fields where multiple stars fall within 1 arcsec

The **32 missing targets** might be due to:
- Sources outside SMASH DR2 footprint
- Sources below SMASH detection limits
- Bad coordinates in original catalog

# Creating Cleaned SMASH Data with Brightest Matches Only

In [27]:
# Examine the SMASH data structure
print("SMASH DR2 data columns:")
print(smash_results.colnames)
print(f"\nTotal columns: {len(smash_results.colnames)}")

# Check for magnitude columns
mag_columns = [col for col in smash_results.colnames if 'mag' in col.lower() or col.lower() in ['u', 'g', 'r', 'i', 'z']]
print(f"\nPotential magnitude columns: {mag_columns}")

# Check specifically for u-band magnitude
u_columns = [col for col in smash_results.colnames if 'u' in col.lower() and 'mag' in col.lower()]
if not u_columns:
    u_columns = [col for col in smash_results.colnames if col.lower() == 'u']
print(f"\nU-band magnitude columns: {u_columns}")

# Show a sample row to understand data types
print(f"\nSample SMASH data (first row):")
sample_row = smash_results[0]
for col in smash_results.colnames[:10]:  # Show first 10 columns
    print(f"  {col}: {sample_row[col]}")
if len(smash_results.colnames) > 10:
    print(f"  ... and {len(smash_results.colnames) - 10} more columns")

SMASH DR2 data columns:
['ra', 'dec', 'glon', 'glat', 'elon', 'elat', 'raerr', 'decerr', 'rascatter', 'decscatter', 'umag', 'uerr', 'uscatter', 'gmag', 'gerr', 'gscatter', 'rmag', 'rerr', 'rscatter', 'imag', 'ierr', 'iscatter', 'zmag', 'zerr', 'zscatter', 'chi', 'sharp', 'prob', 'ebv', 'htm9', 'ring256', 'nest4096', 'random_id', 'ndet', 'depthflag', 'ndetu', 'ndetg', 'ndetr', 'ndeti', 'ndetz', 'flag', 'id']

Total columns: 42

Potential magnitude columns: ['umag', 'gmag', 'rmag', 'imag', 'zmag']

U-band magnitude columns: ['umag']

Sample SMASH data (first row):
  ra: 9.3801432121013
  dec: -74.41106480068613
  glon: 304.20317245578434
  glat: -42.68281797822008
  elon: 307.70744250524353
  elat: -64.31021826987711
  raerr: 0.008372999727725983
  decerr: 0.008372999727725983
  rascatter: 0.014771999791264534
  decscatter: 0.021981999278068542
  ... and 32 more columns


In [28]:
# Create the cleaned dataset with only the brightest U-band matches
print("Creating cleaned SMASH dataset...")
print("=" * 50)

# Initialize the final dataset list
cleaned_data = []

# Process each target
for target_idx in range(len(targets)):
    # Get target information
    target_ra = targets.iloc[target_idx]['RA']
    target_dec = targets.iloc[target_idx]['DEC']
    
    # Create base row with target coordinates and other target data
    row_data = {
        'target_idx': target_idx,
        'target_ra': target_ra,
        'target_dec': target_dec
    }
    
    # Add all other columns from the original target data
    for col in targets.columns:
        if col not in ['RA', 'DEC']:  # Avoid duplicates
            row_data[f'target_{col.lower()}'] = targets.iloc[target_idx][col]
    
    # Check if this target has matches
    if target_idx in target_match_counts:
        smash_indices = target_match_counts[target_idx]
        n_matches = len(smash_indices)
        
        # Set the multiple match flag
        multiple_match_flag = 1 if n_matches > 1 else 0
        row_data['multiple_match_flag'] = multiple_match_flag
        
        if n_matches == 1:
            # Single match - use it directly
            smash_idx = smash_indices[0]
            best_smash_row = smash_results[smash_idx]
        else:
            # Multiple matches - find the brightest (smallest umag)
            best_smash_idx = None
            best_umag = float('inf')
            
            for smash_idx in smash_indices:
                umag = smash_results['umag'][smash_idx]
                if not np.isnan(umag) and umag < best_umag:
                    best_umag = umag
                    best_smash_idx = smash_idx
            
            # If all have NaN umag, just take the first one
            if best_smash_idx is None:
                best_smash_idx = smash_indices[0]
            
            best_smash_row = smash_results[best_smash_idx]
        
        # Add SMASH data to the row
        for col in smash_results.colnames:
            row_data[f'smash_{col}'] = best_smash_row[col]
        
        # Add the distance information
        smash_idx_in_array = np.where(idx_closest_targets == target_idx)[0]
        if len(smash_idx_in_array) > 0:
            # Find the distance for this specific match
            matching_distances = []
            for i, closest_target in enumerate(idx_closest_targets):
                if closest_target == target_idx:
                    matching_distances.append(distances_arcsec[i])
            
            if len(matching_distances) > 0:
                # For multiple matches, we'll report the distance of the brightest match
                if n_matches == 1:
                    row_data['distance_arcsec'] = matching_distances[0]
                else:
                    # Find which distance corresponds to our chosen brightest match
                    for i, smash_idx in enumerate(smash_indices):
                        if smash_idx == best_smash_idx:
                            idx_in_distances = 0
                            for j, closest_target in enumerate(idx_closest_targets):
                                if closest_target == target_idx:
                                    if idx_in_distances == i:
                                        row_data['distance_arcsec'] = distances_arcsec[j]
                                        break
                                    idx_in_distances += 1
                            break
            else:
                row_data['distance_arcsec'] = np.nan
        else:
            row_data['distance_arcsec'] = np.nan
            
    else:
        # No matches found
        row_data['multiple_match_flag'] = 0
        
        # Add NaN values for all SMASH columns
        for col in smash_results.colnames:
            row_data[f'smash_{col}'] = np.nan
            
        row_data['distance_arcsec'] = np.nan
    
    cleaned_data.append(row_data)

print(f"Processed {len(cleaned_data)} targets")

# Convert to DataFrame for easier handling
cleaned_df = pd.DataFrame(cleaned_data)
print(f"Created DataFrame with shape: {cleaned_df.shape}")
print(f"Columns: {list(cleaned_df.columns)}")

# Check the multiple match flag distribution
flag_counts = cleaned_df['multiple_match_flag'].value_counts().sort_index()
print(f"\nMultiple match flag distribution:")
print(f"  0 (single/no matches): {flag_counts.get(0, 0)}")
print(f"  1 (multiple matches): {flag_counts.get(1, 0)}")

Creating cleaned SMASH dataset...
Processed 848 targets
Created DataFrame with shape: (848, 78)
Columns: ['target_idx', 'target_ra', 'target_dec', 'target_overall_mean', 'target_overall_median', 'target_overall_mean_mag', 'target_overall_median_mag', 'target_offset_flag', 'target_chi_squared_68', 'target_chi2_threshold_68', 'target_chi_flag_68', 'target_chi_squared_95', 'target_chi2_threshold_95', 'target_chi_flag_95', 'target_chi_squared_997', 'target_chi2_threshold_997', 'target_chi_flag_997', 'target_amplitude_5', 'target_lower_percentile_5', 'target_upper_percentile_5', 'target_mag_amplitude_5', 'target_amplitude_1', 'target_lower_percentile_1', 'target_upper_percentile_1', 'target_mag_amplitude_1', 'target_largest_amp', 'target_best_period', 'target_alarm_level_flag', 'target_std_amplitude', 'target_logt', 'target_logl', 'target_cephied_like', 'target_ir_variable_like', 'target_other_variable_like', 'multiple_match_flag', 'smash_ra', 'smash_dec', 'smash_glon', 'smash_glat', 'smash

In [29]:
# Save the cleaned dataset as CSV
output_csv_file = 'smash_targets_cleaned.csv'
cleaned_df.to_csv(output_csv_file, index=False)

print(f"Cleaned dataset saved as: {output_csv_file}")
print(f"Shape: {cleaned_df.shape}")
print(f"Columns: {len(cleaned_df.columns)}")

# Quick summary of what we created
print(f"\nSummary of cleaned dataset:")
print(f"- Total targets: {len(cleaned_df)}")
print(f"- Targets with SMASH matches: {(~cleaned_df['smash_ra'].isna()).sum()}")
print(f"- Targets without matches: {cleaned_df['smash_ra'].isna().sum()}")
print(f"- Targets with multiple matches (flag=1): {(cleaned_df['multiple_match_flag'] == 1).sum()}")
print(f"- For multiple matches: selected brightest U-band magnitude")
print(f"- All target data preserved with 'target_' prefix")
print(f"- All SMASH data included with 'smash_' prefix")
print(f"- Distance in arcseconds included")
print(f"- Multiple match flag: 0 = single/no match, 1 = multiple matches")

Cleaned dataset saved as: smash_targets_cleaned.csv
Shape: (848, 78)
Columns: 78

Summary of cleaned dataset:
- Total targets: 848
- Targets with SMASH matches: 816
- Targets without matches: 32
- Targets with multiple matches (flag=1): 154
- For multiple matches: selected brightest U-band magnitude
- All target data preserved with 'target_' prefix
- All SMASH data included with 'smash_' prefix
- Distance in arcseconds included
- Multiple match flag: 0 = single/no match, 1 = multiple matches


In [ ]:
# Validation and summary of the cleaned dataset
print("VALIDATION OF CLEANED DATASET")
print("=" * 40)

# Check matches vs no matches
has_smash_data = ~cleaned_df['smash_ra'].isna()
n_with_matches = has_smash_data.sum()
n_without_matches = (~has_smash_data).sum()

print(f"Targets with SMASH matches: {n_with_matches}")
print(f"Targets without SMASH matches: {n_without_matches}")
print(f"Total targets: {len(cleaned_df)}")
print(f"Verification: {n_with_matches + n_without_matches == len(cleaned_df)}")

# Check the flag distribution again
print(f"\nFlag distribution:")
print(f"  Flag 0 (0 or 1 match): {(cleaned_df['multiple_match_flag'] == 0).sum()}")
print(f"  Flag 1 (2+ matches): {(cleaned_df['multiple_match_flag'] == 1).sum()}")

# Verify that all targets with multiple_match_flag=1 actually have SMASH data
multiple_match_targets = cleaned_df[cleaned_df['multiple_match_flag'] == 1]
all_have_data = (~multiple_match_targets['smash_ra'].isna()).all()
print(f"\nAll multiple-match targets have SMASH data: {all_have_data}")

# Show some examples of the brightest selection
print(f"\nExamples of targets with multiple matches (showing U mag selection):")
examples = cleaned_df[cleaned_df['multiple_match_flag'] == 1].head(3)
for idx, row in examples.iterrows():
    target_idx = int(row['target_idx'])
    if target_idx in target_match_counts:
        original_matches = target_match_counts[target_idx]
        print(f"\nTarget {target_idx} (RA={row['target_ra']:.6f}, DEC={row['target_dec']:.6f}):")
        print(f"  Had {len(original_matches)} matches")
        print(f"  Selected U mag: {row['smash_umag']:.3f}")
        print(f"  Selected SMASH coordinates: RA={row['smash_ra']:.6f}, DEC={row['smash_dec']:.6f}")
        print(f"  Distance: {row['distance_arcsec']:.3f} arcsec")

# Check for any issues with U magnitude selection
problem_cases = 0
for target_idx in range(len(targets)):
    if target_idx in target_match_counts:
        smash_indices = target_match_counts[target_idx]
        if len(smash_indices) > 1:
            umags = [smash_results['umag'][i] for i in smash_indices]
            selected_umag = cleaned_df.loc[cleaned_df['target_idx'] == target_idx, 'smash_umag'].iloc[0]
            
            # Check if we selected the brightest available
            finite_umags = [u for u in umags if not np.isnan(u)]
            if finite_umags and not np.isnan(selected_umag):
                brightest_available = min(finite_umags)
                if abs(selected_umag - brightest_available) > 1e-6:
                    problem_cases += 1

print(f"\nBrightest selection verification: {problem_cases} problem cases found")

In [30]:
# Display all targets with multiple matches and their U-band magnitudes
print("TARGETS WITH MULTIPLE MATCHES - U-BAND MAGNITUDE COMPARISON")
print("=" * 70)
print(f"Total targets with multiple matches: {len([t for t, matches in target_match_counts.items() if len(matches) > 1])}")
print("=" * 70)

multiple_targets = []
for target_idx, smash_indices in target_match_counts.items():
    if len(smash_indices) > 1:
        multiple_targets.append((target_idx, smash_indices))

# Sort by target index for easier scrolling
multiple_targets.sort(key=lambda x: x[0])

for i, (target_idx, smash_indices) in enumerate(multiple_targets, 1):
    target_ra = targets.iloc[target_idx]['RA']
    target_dec = targets.iloc[target_idx]['DEC']
    
    print(f"\n[{i:3d}] TARGET {target_idx:3d}: RA={target_ra:9.6f}, DEC={target_dec:9.6f}")
    print(f"     {len(smash_indices)} matches found:")
    
    # Collect all match info for sorting
    match_info = []
    for j, smash_idx in enumerate(smash_indices):
        smash_ra = smash_results['ra'][smash_idx]
        smash_dec = smash_results['dec'][smash_idx]
        umag = smash_results['umag'][smash_idx]
        distance = distances_arcsec[np.where(idx_closest_targets == target_idx)[0][j]] if j < len(np.where(idx_closest_targets == target_idx)[0]) else np.nan
        
        match_info.append({
            'smash_idx': smash_idx,
            'ra': smash_ra,
            'dec': smash_dec,
            'umag': umag,
            'distance': distance
        })
    
    # Sort by U magnitude (brightest first, NaN last)
    match_info.sort(key=lambda x: (np.isnan(x['umag']), x['umag']))
    
    # Display sorted matches
    for j, match in enumerate(match_info):
        umag_str = f"{match['umag']:6.3f}" if not np.isnan(match['umag']) else " NaN  "
        brightest_marker = " ← BRIGHTEST" if j == 0 and not np.isnan(match['umag']) else ""
        selected_marker = " ← SELECTED" if match['smash_idx'] == smash_indices[0] else ""
        
        print(f"     [{j+1}] SMASH {match['smash_idx']:4d}: RA={match['ra']:9.6f}, DEC={match['dec']:9.6f}, "
              f"U={umag_str}, dist={match['distance']:5.3f}\"{brightest_marker}{selected_marker}")
    
    # Show which one was actually selected in the cleaned dataset
    selected_umag = cleaned_df.loc[cleaned_df['target_idx'] == target_idx, 'smash_umag'].iloc[0]
    print(f"     → Selected U mag: {selected_umag:.3f}")
    
    if (i % 10) == 0:  # Add separator every 10 targets for easier reading
        print("-" * 70)

print(f"\n{'='*70}")
print(f"Displayed {len(multiple_targets)} targets with multiple matches")
print("Note: Matches sorted by U magnitude (brightest first)")
print("'SELECTED' shows what was chosen for the cleaned dataset")

TARGETS WITH MULTIPLE MATCHES - U-BAND MAGNITUDE COMPARISON
Total targets with multiple matches: 154

[  1] TARGET   2: RA= 7.925977, DEC=-73.531670
     2 matches found:
     [1] SMASH   18: RA= 7.925979, DEC=-73.531678, U=17.539, dist=0.028" ← BRIGHTEST
     [2] SMASH   17: RA= 7.926168, DEC=-73.531815, U=21.352, dist=0.557" ← SELECTED
     → Selected U mag: 17.539

[  2] TARGET   4: RA= 7.980047, DEC=-73.578545
     4 matches found:
     [1] SMASH   15: RA= 7.980068, DEC=-73.578550, U=18.296, dist=0.027" ← BRIGHTEST
     [2] SMASH   14: RA= 7.980073, DEC=-73.578656, U=21.589, dist=0.400"
     [3] SMASH   13: RA= 7.979233, DEC=-73.578518, U=99.990, dist=0.834" ← SELECTED
     [4] SMASH   16: RA= 7.980707, DEC=-73.578618, U=99.990, dist=0.722"
     → Selected U mag: 18.296

[  3] TARGET   5: RA= 7.981742, DEC=-73.585571
     4 matches found:
     [1] SMASH   10: RA= 7.981785, DEC=-73.585581, U=17.360, dist=0.057" ← BRIGHTEST
     [2] SMASH    9: RA= 7.981183, DEC=-73.585696, U=99.990,

/var/folders/34/3847lqd14j78mfm70_z1c_r00000gn/T/ipykernel_42730/2796537658.py:43: FutureWarning: Format strings passed to MaskedConstant are ignored, but in future may error or produce different behavior
  umag_str = f"{match['umag']:6.3f}" if not np.isnan(match['umag']) else " NaN  "
/var/folders/34/3847lqd14j78mfm70_z1c_r00000gn/T/ipykernel_42730/2796537658.py:52: FutureWarning: Format strings passed to MaskedConstant are ignored, but in future may error or produce different behavior
  print(f"     → Selected U mag: {selected_umag:.3f}")


# Corrected Brightest U-band Selection

In [33]:
# Let's debug the U-band selection issue systematically
print("DEBUGGING U-BAND SELECTION ISSUES")
print("=" * 50)

# Find targets with multiple matches
targets_with_multiples = [t for t, matches in target_match_counts.items() if len(matches) > 1]
print(f"Found {len(targets_with_multiples)} targets with multiple matches")

# Check the first few targets with multiple matches
for i, target_idx in enumerate(targets_with_multiples[:3]):
    smash_indices = target_match_counts[target_idx]
    print(f"\nDEBUG Target {target_idx} (has {len(smash_indices)} matches):")
    
    umag_values = []
    for j, smash_idx in enumerate(smash_indices):
        umag_raw = smash_results['umag'][smash_idx]
        # Handle masked values properly
        if hasattr(umag_raw, 'mask') and np.ma.is_masked(umag_raw):
            umag_float = np.nan
        else:
            umag_float = float(umag_raw)
        
        umag_values.append(umag_float)
        print(f"  Match {j+1}: SMASH {smash_idx}, umag={umag_float:.3f}")
    
    # Find the brightest (smallest finite umag)
    finite_umags = [u for u in umag_values if np.isfinite(u)]
    if finite_umags:
        brightest_umag = min(finite_umags)
        brightest_idx = umag_values.index(brightest_umag)
        print(f"  → Brightest should be Match {brightest_idx+1} with umag={brightest_umag:.3f}")
    else:
        print(f"  → No finite umag values!")
    
    # Check what was actually selected in the cleaned dataset
    selected_umag = cleaned_df.loc[cleaned_df['target_idx'] == target_idx, 'smash_umag'].iloc[0]
    print(f"  → Actually selected: umag={selected_umag:.3f}")
    
    if finite_umags and abs(selected_umag - brightest_umag) > 1e-3:
        print(f"  *** PROBLEM: Wrong selection! Should be {brightest_umag:.3f}, got {selected_umag:.3f}")
        
print("\nThis will help us understand the selection issue...")

DEBUGGING U-BAND SELECTION ISSUES
Found 154 targets with multiple matches

DEBUG Target 5 (has 4 matches):
  Match 1: SMASH 9, umag=99.990
  Match 2: SMASH 10, umag=17.360
  Match 3: SMASH 11, umag=99.990
  Match 4: SMASH 12, umag=99.990
  → Brightest should be Match 2 with umag=17.360
  → Actually selected: umag=17.360

DEBUG Target 4 (has 4 matches):
  Match 1: SMASH 13, umag=99.990
  Match 2: SMASH 14, umag=21.589
  Match 3: SMASH 15, umag=18.296
  Match 4: SMASH 16, umag=99.990
  → Brightest should be Match 3 with umag=18.296
  → Actually selected: umag=18.296

DEBUG Target 2 (has 2 matches):
  Match 1: SMASH 17, umag=21.352
  Match 2: SMASH 18, umag=17.539
  → Brightest should be Match 2 with umag=17.539
  → Actually selected: umag=17.539

This will help us understand the selection issue...


In [34]:
# Let's systematically check ALL targets with multiple matches to find problematic ones
print("SYSTEMATIC CHECK FOR INCORRECT SELECTIONS")
print("=" * 50)

problem_count = 0
total_checked = 0

for target_idx in targets_with_multiples:
    smash_indices = target_match_counts[target_idx]
    total_checked += 1
    
    # Get all umag values for this target's matches
    umag_values = []
    for smash_idx in smash_indices:
        umag_raw = smash_results['umag'][smash_idx]
        if hasattr(umag_raw, 'mask') and np.ma.is_masked(umag_raw):
            umag_float = np.nan
        else:
            umag_float = float(umag_raw)
        umag_values.append(umag_float)
    
    # Find the brightest (smallest finite umag)
    finite_umags = [u for u in umag_values if np.isfinite(u)]
    
    if finite_umags:
        brightest_umag = min(finite_umags)
        
        # Check what was actually selected
        selected_umag = cleaned_df.loc[cleaned_df['target_idx'] == target_idx, 'smash_umag'].iloc[0]
        
        # Check for problems (allowing small numerical precision errors)
        if np.isfinite(selected_umag) and abs(selected_umag - brightest_umag) > 1e-3:
            problem_count += 1
            if problem_count <= 5:  # Show first 5 problem cases
                print(f"\nPROBLEM Target {target_idx}:")
                print(f"  All umag values: {[f'{u:.3f}' if np.isfinite(u) else 'NaN' for u in umag_values]}")
                print(f"  Brightest should be: {brightest_umag:.3f}")
                print(f"  Actually selected: {selected_umag:.3f}")

print(f"\nSUMMARY:")
print(f"Checked {total_checked} targets with multiple matches")
print(f"Found {problem_count} targets with incorrect brightest selection")

if problem_count > 0:
    print(f"\nThere ARE issues with the brightest selection! Let's fix this.")
else:
    print(f"\nNo issues found - the selection logic appears correct!")
    print("The display issue might be elsewhere...")

SYSTEMATIC CHECK FOR INCORRECT SELECTIONS

SUMMARY:
Checked 154 targets with multiple matches
Found 0 targets with incorrect brightest selection

No issues found - the selection logic appears correct!
The display issue might be elsewhere...


In [40]:
# Find rows with U-band magnitude values around 99 (indicating bad/missing measurements)
print("SEARCHING FOR ROWS WITH ~99 U-BAND MAGNITUDE VALUES")
print("=" * 60)

# Load the cleaned dataset (in case it's not in memory)
if 'cleaned_df' not in globals():
    cleaned_df = pd.read_csv('smash_targets_cleaned.csv')

print(f"Loaded cleaned dataset with {len(cleaned_df)} rows")

# Focus only on U-band magnitude
u_mag_column = 'smash_umag'
print(f"Checking U-band magnitude column: {u_mag_column}")

# Check for values around 99 (between 99 and 100 to catch 99.9, 99.99, etc.)
bad_umag_rows = []

for idx, row in cleaned_df.iterrows():
    # Check only U-band magnitude for values around 99
    umag_value = row[u_mag_column]
    # Handle different data types more carefully
    try:
        if not pd.isna(umag_value):
            umag_float = float(umag_value)
            if 99.0 <= umag_float <= 100.0:
                bad_umag_rows.append({
                    'row_idx': idx,
                    'target_idx': row['target_idx'],
                    'target_ra': row['target_ra'],
                    'target_dec': row['target_dec'],
                    'multiple_match_flag': row['multiple_match_flag'],
                    'umag': umag_float
                })
    except (TypeError, ValueError):
        # Skip problematic values
        continue

print(f"\nFound {len(bad_umag_rows)} rows with U-band magnitude values around 99")

# Display the problematic rows
if len(bad_umag_rows) > 0:
    print("\nROWS WITH ~99 U-BAND MAGNITUDE VALUES:")
    print("-" * 60)
    
    for i, bad_row in enumerate(bad_umag_rows):
        print(f"\n[{i+1:3d}] Target {bad_row['target_idx']:3d}: "
              f"RA={bad_row['target_ra']:9.6f}, DEC={bad_row['target_dec']:9.6f}")
        print(f"     Multiple matches: {'Yes' if bad_row['multiple_match_flag'] == 1 else 'No'}")
        print(f"     U-band magnitude: {bad_row['umag']:.3f}")
        
        if (i + 1) % 20 == 0:  # Add separator every 20 rows
            print("-" * 40)
            
else:
    print("\nNo rows found with U-band magnitude values around 99!")

print(f"\n{'='*60}")
print(f"Summary: {len(bad_umag_rows)} targets have U-band magnitude values ~99")
if len(bad_umag_rows) > 0:
    multiple_match_bad = sum(1 for row in bad_umag_rows if row['multiple_match_flag'] == 1)
    print(f"Of these, {multiple_match_bad} had multiple matches (questionable selection)")
    print(f"And {len(bad_umag_rows) - multiple_match_bad} had single/no matches")

SEARCHING FOR ROWS WITH ~99 U-BAND MAGNITUDE VALUES
Loaded cleaned dataset with 848 rows
Checking U-band magnitude column: smash_umag

Found 0 rows with U-band magnitude values around 99

No rows found with U-band magnitude values around 99!

Summary: 0 targets have U-band magnitude values ~99


/var/folders/34/3847lqd14j78mfm70_z1c_r00000gn/T/ipykernel_42730/3514756092.py:24: UserWarning: Warning: converting a masked element to nan.
  umag_float = float(umag_value)


In [41]:
# Add SMASH U magnitude and error to Anna's candidate files
print("ADDING SMASH U MAGNITUDES TO CANDIDATE FILES")
print("=" * 60)

# Load the cleaned SMASH data
if 'cleaned_df' not in globals():
    cleaned_df = pd.read_csv('smash_targets_cleaned.csv')

print(f"Loaded cleaned SMASH data with {len(cleaned_df)} targets")

# File paths to update
candidate_files = [
    './annas_candidates/final_lmc_ysgcands.csv',
    './annas_candidates/final_smc_ysgcands.csv'
]

# Function to match coordinates with tolerance
def find_closest_smash_match(ra, dec, smash_data, tolerance_arcsec=1.0):
    """Find the closest SMASH match within tolerance"""
    # Convert tolerance from arcseconds to degrees
    tolerance_deg = tolerance_arcsec / 3600.0
    
    # Calculate distances
    distances = np.sqrt((smash_data['target_ra'] - ra)**2 + 
                       (smash_data['target_dec'] - dec)**2)
    
    # Find closest match within tolerance
    closest_idx = np.argmin(distances)
    closest_distance = distances.iloc[closest_idx]
    
    if closest_distance <= tolerance_deg:
        return smash_data.iloc[closest_idx]
    else:
        return None

# Process each candidate file
for file_path in candidate_files:
    print(f"\nProcessing: {file_path}")
    
    try:
        # Load the candidate file
        candidates = pd.read_csv(file_path)
        print(f"  Loaded {len(candidates)} candidates")
        print(f"  Original columns: {list(candidates.columns)}")
        
        # Identify RA/DEC columns (try common column names)
        ra_col = None
        dec_col = None
        
        # Check for common RA/DEC column names
        for col in candidates.columns:
            if col.upper() in ['RA', '_RA', 'RA_DEG', 'RADEG']:
                ra_col = col
            elif col.upper() in ['DEC', '_DEC', 'DEC_DEG', 'DECDEG', 'DECL']:
                dec_col = col
        
        # If not found, try partial matches
        if ra_col is None:
            ra_candidates = [col for col in candidates.columns if 'ra' in col.lower()]
            if ra_candidates:
                ra_col = ra_candidates[0]
                
        if dec_col is None:
            dec_candidates = [col for col in candidates.columns if 'dec' in col.lower()]
            if dec_candidates:
                dec_col = dec_candidates[0]
        
        if ra_col is None or dec_col is None:
            print(f"  ERROR: Could not identify RA/DEC columns in {file_path}")
            print(f"  Available columns: {list(candidates.columns)}")
            continue
            
        print(f"  Using RA column: {ra_col}")
        print(f"  Using DEC column: {dec_col}")
        
        # Initialize new columns with NaN
        candidates['Usmashmag'] = np.nan
        candidates['e_Usmashmag'] = np.nan
        
        # Match coordinates and add U magnitude data
        matches_found = 0
        for idx, row in candidates.iterrows():
            ra = row[ra_col]
            dec = row[dec_col]
            
            # Find closest SMASH match
            smash_match = find_closest_smash_match(ra, dec, cleaned_df)
            
            if smash_match is not None:
                # Check if SMASH data has valid U magnitude
                if not pd.isna(smash_match['smash_umag']):
                    candidates.at[idx, 'Usmashmag'] = smash_match['smash_umag']
                
                if not pd.isna(smash_match['smash_uerr']):
                    candidates.at[idx, 'e_Usmashmag'] = smash_match['smash_uerr']
                
                matches_found += 1
        
        print(f"  Found SMASH matches for {matches_found}/{len(candidates)} candidates")
        
        # Count how many got U magnitude data
        with_umag = (~candidates['Usmashmag'].isna()).sum()
        with_uerr = (~candidates['e_Usmashmag'].isna()).sum()
        print(f"  Added U magnitude to {with_umag} candidates")
        print(f"  Added U magnitude error to {with_uerr} candidates")
        
        # Save the updated file
        candidates.to_csv(file_path, index=False)
        print(f"  Updated file saved: {file_path}")
        print(f"  New columns: {list(candidates.columns)}")
        
    except FileNotFoundError:
        print(f"  ERROR: File not found: {file_path}")
    except Exception as e:
        print(f"  ERROR processing {file_path}: {e}")

print(f"\n{'='*60}")
print("COMPLETED: Added SMASH U magnitudes to candidate files")
print("New columns added: 'Usmashmag', 'e_Usmashmag'")

ADDING SMASH U MAGNITUDES TO CANDIDATE FILES
Loaded cleaned SMASH data with 848 targets

Processing: ./annas_candidates/final_lmc_ysgcands.csv
  Loaded 471 candidates
  Original columns: ['ra', 'dec', '2MASS', 'ra_gaia', 'dec_gaia', 'parallax', 'pmra', 'pmdec', 'pm', 'ra_error', 'dec_error', 'parallax_error', 'parallax_over_error', 'pmra_error', 'pmdec_error', 'astrometric_gof_al', 'astrometric_chi2_al', 'astrometric_excess_noise', 'astrometric_excess_noise_sig', 'ruwe', 'astrometric_params_solved', 'astrometric_sigma5d_max', 'duplicated_source', 'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'phot_g_mean_flux', 'phot_bp_mean_flux', 'phot_rp_mean_flux', 'phot_g_mean_flux_error', 'phot_g_mean_flux_over_error', 'phot_bp_mean_flux_error', 'phot_bp_mean_flux_over_error', 'phot_rp_mean_flux_error', 'phot_rp_mean_flux_over_error', 'phot_bp_rp_excess_factor', 'Separation_1', 'covariance', 'Jmag', 'Hmag', 'Kmag', 'e_Jmag', 'e_Hmag', 'e_Kmag', 'Qfl', 'Umag', 'e_Umag', 'Bmag', 'e_Bma

/opt/anaconda3/envs/datalab/lib/python3.14/site-packages/pandas/core/internals/managers.py:2209: UserWarning: Warning: converting a masked element to nan.
  arr[indexer] = value


In [43]:
# Check how many rows have valid Usmashmag values in the candidate files
print("CHECKING VALID USMASHMAG VALUES IN CANDIDATE FILES")
print("=" * 60)

candidate_files = [
    './annas_candidates/final_lmc_ysgcands.csv',
    './annas_candidates/final_smc_ysgcands.csv'
]

total_candidates = 0
total_with_usmashmag = 0
total_with_uerr = 0

for file_path in candidate_files:
    print(f"\nFile: {file_path}")
    
    try:
        # Load the candidate file
        df = pd.read_csv(file_path)
        n_candidates = len(df)
        
        # Count valid Usmashmag values
        valid_usmashmag = (~df['Usmashmag'].isna()).sum()
        valid_uerr = (~df['e_Usmashmag'].isna()).sum()
        
        print(f"  Total candidates: {n_candidates}")
        print(f"  With valid Usmashmag: {valid_usmashmag} ({100*valid_usmashmag/n_candidates:.1f}%)")
        print(f"  With valid e_Usmashmag: {valid_uerr} ({100*valid_uerr/n_candidates:.1f}%)")
        
        # Add to totals
        total_candidates += n_candidates
        total_with_usmashmag += valid_usmashmag
        total_with_uerr += valid_uerr
        
    except FileNotFoundError:
        print(f"  ERROR: File not found: {file_path}")
    except Exception as e:
        print(f"  ERROR: {e}")

print(f"\n{'='*60}")
print("SUMMARY ACROSS ALL CANDIDATE FILES:")
print(f"Total candidates: {total_candidates}")
print(f"With valid Usmashmag: {total_with_usmashmag} ({100*total_with_usmashmag/total_candidates:.1f}%)")
print(f"With valid e_Usmashmag: {total_with_uerr} ({100*total_with_uerr/total_candidates:.1f}%)")
print(f"Missing SMASH U magnitudes: {total_candidates - total_with_usmashmag}")

CHECKING VALID USMASHMAG VALUES IN CANDIDATE FILES

File: ./annas_candidates/final_lmc_ysgcands.csv
  Total candidates: 471
  With valid Usmashmag: 433 (91.9%)
  With valid e_Usmashmag: 439 (93.2%)

File: ./annas_candidates/final_smc_ysgcands.csv
  Total candidates: 377
  With valid Usmashmag: 377 (100.0%)
  With valid e_Usmashmag: 377 (100.0%)

SUMMARY ACROSS ALL CANDIDATE FILES:
Total candidates: 848
With valid Usmashmag: 810 (95.5%)
With valid e_Usmashmag: 816 (96.2%)
Missing SMASH U magnitudes: 38


In [45]:
# Investigate why some rows have U magnitude error but no U magnitude
print("INVESTIGATING U MAGNITUDE vs U MAGNITUDE ERROR DISCREPANCY")
print("=" * 70)

# Check both candidate files
candidate_files = [
    './annas_candidates/final_lmc_ysgcands.csv',
    './annas_candidates/final_smc_ysgcands.csv'
]

total_problem_cases = 0

for file_path in candidate_files:
    print(f"\nAnalyzing: {file_path}")
    
    try:
        df = pd.read_csv(file_path)
        
        # Check for cases where error exists but magnitude doesn't
        has_uerr_no_umag = (~df['e_Usmashmag'].isna()) & (df['Usmashmag'].isna())
        n_problem_cases = has_uerr_no_umag.sum()
        total_problem_cases += n_problem_cases
        
        print(f"  Total candidates: {len(df)}")
        print(f"  With valid Usmashmag: {(~df['Usmashmag'].isna()).sum()}")
        print(f"  With valid e_Usmashmag: {(~df['e_Usmashmag'].isna()).sum()}")
        print(f"  ERROR cases (has e_Usmashmag but no Usmashmag): {n_problem_cases}")
        
        if n_problem_cases > 0:
            print(f"  Problem rows:")
            problem_rows = df[has_uerr_no_umag]
            for idx, row in problem_rows.head(5).iterrows():
                print(f"    Row {idx}: Usmashmag={row['Usmashmag']}, e_Usmashmag={row['e_Usmashmag']:.3f}")
            
            # Let's trace back to the SMASH data for these cases
            print(f"\n  Investigating source of problem in cleaned SMASH data...")
            
            # Use the same coordinate matching to find the SMASH sources
            for idx, row in problem_rows.head(3).iterrows():
                # Find the corresponding RA/DEC columns
                ra_col = None
                dec_col = None
                for col in df.columns:
                    if 'ra' in col.lower() and ra_col is None:
                        ra_col = col
                    elif 'dec' in col.lower() and dec_col is None:
                        dec_col = col
                
                if ra_col and dec_col:
                    ra = row[ra_col]
                    dec = row[dec_col]
                    
                    # Find the matching cleaned SMASH data
                    tolerance_deg = 1.0 / 3600.0  # 1 arcsec
                    distances = np.sqrt((cleaned_df['target_ra'] - ra)**2 + 
                                      (cleaned_df['target_dec'] - dec)**2)
                    closest_idx = np.argmin(distances)
                    
                    if distances.iloc[closest_idx] <= tolerance_deg:
                        smash_row = cleaned_df.iloc[closest_idx]
                        print(f"    Problem case at row {idx}:")
                        print(f"      SMASH smash_umag: {smash_row['smash_umag']}")
                        print(f"      SMASH smash_uerr: {smash_row['smash_uerr']}")
                        print(f"      pd.isna(smash_umag): {pd.isna(smash_row['smash_umag'])}")
                        print(f"      pd.isna(smash_uerr): {pd.isna(smash_row['smash_uerr'])}")
                        
                        # Check the raw values
                        print(f"      Raw smash_umag type: {type(smash_row['smash_umag'])}")
                        print(f"      Raw smash_uerr type: {type(smash_row['smash_uerr'])}")
                        
                        if hasattr(smash_row['smash_umag'], 'mask'):
                            print(f"      smash_umag is masked: {np.ma.is_masked(smash_row['smash_umag'])}")
                        if hasattr(smash_row['smash_uerr'], 'mask'):
                            print(f"      smash_uerr is masked: {np.ma.is_masked(smash_row['smash_uerr'])}")
                    
    except FileNotFoundError:
        print(f"  ERROR: File not found: {file_path}")
    except Exception as e:
        print(f"  ERROR: {e}")

print(f"\n{'='*70}")
print(f"TOTAL PROBLEM CASES ACROSS ALL FILES: {total_problem_cases}")

if total_problem_cases > 0:
    print("\nThis suggests an issue in the data assignment logic where:")
    print("1. SMASH U magnitude is NaN/masked but error is not")
    print("2. The pd.isna() check behaves differently for magnitude vs error")
    print("3. There might be inconsistency in how masked values are handled")

INVESTIGATING U MAGNITUDE vs U MAGNITUDE ERROR DISCREPANCY

Analyzing: ./annas_candidates/final_lmc_ysgcands.csv
  Total candidates: 471
  With valid Usmashmag: 433
  With valid e_Usmashmag: 439
  ERROR cases (has e_Usmashmag but no Usmashmag): 6
  Problem rows:
    Row 0: Usmashmag=nan, e_Usmashmag=9.990
    Row 2: Usmashmag=nan, e_Usmashmag=9.990
    Row 51: Usmashmag=nan, e_Usmashmag=9.990
    Row 65: Usmashmag=nan, e_Usmashmag=9.990
    Row 80: Usmashmag=nan, e_Usmashmag=9.990

  Investigating source of problem in cleaned SMASH data...
    Problem case at row 0:
      SMASH smash_umag: --
      SMASH smash_uerr: 9.989999771118164
      pd.isna(smash_umag): --
      pd.isna(smash_uerr): False
      Raw smash_umag type: <class 'numpy.ma.core.MaskedConstant'>
      Raw smash_uerr type: <class 'numpy.float64'>
      smash_umag is masked: True
    Problem case at row 2:
      SMASH smash_umag: --
      SMASH smash_uerr: 9.989999771118164
      pd.isna(smash_umag): --
      pd.isna(smash

In [46]:
# Fix the U magnitude assignment logic to properly handle masked values
print("FIXING THE U MAGNITUDE ASSIGNMENT LOGIC")
print("=" * 50)

# Function to check if a value is valid (not NaN and not masked)
def is_valid_value(value):
    """Check if a value is valid (not NaN, not masked)"""
    if pd.isna(value):
        return False
    if hasattr(value, 'mask') and np.ma.is_masked(value):
        return False
    # Check for masked constant
    if str(value) == '--':
        return False
    return True

# Re-read the cleaned SMASH data to ensure we have the right data types
print("Re-checking cleaned SMASH data types...")
print(f"Cleaned SMASH data shape: {cleaned_df.shape}")

# Check some examples of the problematic values
sample_masked = cleaned_df[cleaned_df['smash_uerr'] == 9.989999771118164].head(3)
print(f"\nSample rows with 9.99 errors:")
for idx, row in sample_masked.iterrows():
    print(f"  Row {idx}: smash_umag={row['smash_umag']}, type={type(row['smash_umag'])}")
    print(f"            smash_uerr={row['smash_uerr']}, type={type(row['smash_uerr'])}")
    print(f"            is_valid_umag: {is_valid_value(row['smash_umag'])}")
    print(f"            is_valid_uerr: {is_valid_value(row['smash_uerr'])}")

# Let's fix the LMC candidates file
lmc_file = './annas_candidates/final_lmc_ysgcands.csv'
lmc_df = pd.read_csv(lmc_file)

print(f"\nFixing {lmc_file}...")

# Find problematic rows (have error but no magnitude)
problem_mask = (~lmc_df['e_Usmashmag'].isna()) & (lmc_df['Usmashmag'].isna())
print(f"Found {problem_mask.sum()} problematic rows")

# Clear the error values for these rows since the magnitudes are masked/invalid
lmc_df.loc[problem_mask, 'e_Usmashmag'] = np.nan

# Save the corrected file
lmc_df.to_csv(lmc_file, index=False)
print(f"Corrected {problem_mask.sum()} rows in {lmc_file}")

# Verify the fix
final_problem_count = ((~lmc_df['e_Usmashmag'].isna()) & (lmc_df['Usmashmag'].isna())).sum()
print(f"Remaining problematic rows: {final_problem_count}")

print(f"\nFinal statistics for {lmc_file}:")
print(f"  Total candidates: {len(lmc_df)}")
print(f"  With valid Usmashmag: {(~lmc_df['Usmashmag'].isna()).sum()}")
print(f"  With valid e_Usmashmag: {(~lmc_df['e_Usmashmag'].isna()).sum()}")
print(f"  Consistent magnitude/error pairs: {((~lmc_df['Usmashmag'].isna()) & (~lmc_df['e_Usmashmag'].isna())).sum()}")
print(f"  No magnitude, no error: {(lmc_df['Usmashmag'].isna() & lmc_df['e_Usmashmag'].isna()).sum()}")

print("\n" + "="*50)
print("EXPLANATION: The issue was that SMASH data contains masked values")
print("(numpy.ma.MaskedConstant) for invalid U magnitudes, but pd.isna()")
print("doesn't recognize these as missing values. The error values (9.99)")
print("were valid floats, creating the inconsistency.")
print("Fixed by removing error values where magnitude is masked/invalid.")

FIXING THE U MAGNITUDE ASSIGNMENT LOGIC
Re-checking cleaned SMASH data types...
Cleaned SMASH data shape: (848, 78)

Sample rows with 9.99 errors:
  Row 377: smash_umag=--, type=<class 'numpy.ma.core.MaskedConstant'>
            smash_uerr=9.989999771118164, type=<class 'float'>
            is_valid_umag: False
            is_valid_uerr: True
  Row 379: smash_umag=--, type=<class 'numpy.ma.core.MaskedConstant'>
            smash_uerr=9.989999771118164, type=<class 'float'>
            is_valid_umag: False
            is_valid_uerr: True
  Row 428: smash_umag=--, type=<class 'numpy.ma.core.MaskedConstant'>
            smash_uerr=9.989999771118164, type=<class 'float'>
            is_valid_umag: False
            is_valid_uerr: True

Fixing ./annas_candidates/final_lmc_ysgcands.csv...
Found 6 problematic rows
Corrected 6 rows in ./annas_candidates/final_lmc_ysgcands.csv
Remaining problematic rows: 0

Final statistics for ./annas_candidates/final_lmc_ysgcands.csv:
  Total candidates: 471


In [47]:
# Improved data assignment function for future use
print("CREATING IMPROVED DATA ASSIGNMENT FUNCTION")
print("=" * 50)

def assign_smash_data_improved(candidates_df, smash_df, tolerance_arcsec=1.0):
    """
    Assign SMASH data to candidates with improved masked value handling.
    
    Parameters:
    -----------
    candidates_df : pd.DataFrame
        DataFrame with candidate data containing RA/DEC columns
    smash_df : pd.DataFrame
        Cleaned SMASH data with target coordinates and photometry
    tolerance_arcsec : float
        Matching tolerance in arcseconds (default: 1.0)
    
    Returns:
    --------
    pd.DataFrame : Updated candidates DataFrame with SMASH data
    """
    
    def is_valid_value(value):
        """Check if a value is valid (not NaN, not masked, not sentinel value)"""
        # Check for NaN
        if pd.isna(value):
            return False
        # Check for masked values
        if hasattr(value, 'mask') and np.ma.is_masked(value):
            return False
        # Check for masked constant (shows as '--')
        if str(value) == '--':
            return False
        # Check for sentinel values (like 99.0 or 9.99)
        if isinstance(value, (int, float)) and (abs(value - 99.0) < 0.01 or abs(value - 9.99) < 0.01):
            return False
        return True
    
    # Make a copy to avoid modifying the original
    candidates = candidates_df.copy()
    
    # Initialize SMASH columns if they don't exist
    if 'Usmashmag' not in candidates.columns:
        candidates['Usmashmag'] = np.nan
    if 'e_Usmashmag' not in candidates.columns:
        candidates['e_Usmashmag'] = np.nan
    
    # Find RA/DEC columns in candidates
    ra_col = None
    dec_col = None
    for col in candidates.columns:
        if 'ra' in col.lower() and ra_col is None:
            ra_col = col
        elif 'dec' in col.lower() and dec_col is None:
            dec_col = col
    
    if ra_col is None or dec_col is None:
        print(f"ERROR: Could not find RA/DEC columns. Available: {list(candidates.columns)}")
        return candidates
    
    print(f"Using columns: {ra_col} (RA), {dec_col} (DEC)")
    
    # Convert tolerance to degrees
    tolerance_deg = tolerance_arcsec / 3600.0
    
    matches = 0
    assignments = 0
    
    for idx, candidate in candidates.iterrows():
        ra = candidate[ra_col]
        dec = candidate[dec_col]
        
        if pd.isna(ra) or pd.isna(dec):
            continue
            
        # Find closest SMASH match
        distances = np.sqrt((smash_df['target_ra'] - ra)**2 + 
                          (smash_df['target_dec'] - dec)**2)
        closest_idx = np.argmin(distances)
        
        if distances.iloc[closest_idx] <= tolerance_deg:
            matches += 1
            smash_match = smash_df.iloc[closest_idx]
            
            # Only assign magnitude and error if BOTH are valid
            umag_valid = is_valid_value(smash_match['smash_umag'])
            uerr_valid = is_valid_value(smash_match['smash_uerr'])
            
            if umag_valid and uerr_valid:
                candidates.at[idx, 'Usmashmag'] = float(smash_match['smash_umag'])
                candidates.at[idx, 'e_Usmashmag'] = float(smash_match['smash_uerr'])
                assignments += 1
            else:
                # Ensure both are NaN if either is invalid
                candidates.at[idx, 'Usmashmag'] = np.nan
                candidates.at[idx, 'e_Usmashmag'] = np.nan
    
    print(f"Coordinate matches found: {matches}")
    print(f"Valid U magnitude assignments: {assignments}")
    print(f"Rejected due to masked/invalid values: {matches - assignments}")
    
    return candidates

print("Function created successfully!")
print("\nThis improved function ensures that magnitude and error are assigned")
print("as a consistent pair, preventing the masked value issue we just fixed.")

# Test the function logic with our problem cases
print("\nTesting with problem cases from cleaned SMASH data:")
problem_rows = cleaned_df[cleaned_df['smash_uerr'] == 9.989999771118164].head(3)
for idx, row in problem_rows.iterrows():
    umag_valid = is_valid_value(row['smash_umag'])
    uerr_valid = is_valid_value(row['smash_uerr'])
    print(f"  Row {idx}: umag_valid={umag_valid}, uerr_valid={uerr_valid} -> Would assign: {umag_valid and uerr_valid}")

CREATING IMPROVED DATA ASSIGNMENT FUNCTION
Function created successfully!

This improved function ensures that magnitude and error are assigned
as a consistent pair, preventing the masked value issue we just fixed.

Testing with problem cases from cleaned SMASH data:
  Row 377: umag_valid=False, uerr_valid=True -> Would assign: False
  Row 379: umag_valid=False, uerr_valid=True -> Would assign: False
  Row 428: umag_valid=False, uerr_valid=True -> Would assign: False


In [49]:
# Final verification of both candidate files
print("FINAL VERIFICATION OF CANDIDATE FILES")
print("=" * 50)

candidate_files = [
    './annas_candidates/final_lmc_ysgcands.csv',
    './annas_candidates/final_smc_ysgcands.csv'
]

total_candidates = 0
total_with_smash = 0

for file_path in candidate_files:
    print(f"\nVerifying: {file_path}")
    
    df = pd.read_csv(file_path)
    
    # Check for data consistency
    has_umag = ~df['Usmashmag'].isna()
    has_uerr = ~df['e_Usmashmag'].isna()
    
    consistent_pairs = has_umag == has_uerr
    inconsistent_count = (~consistent_pairs).sum()
    
    total_candidates += len(df)
    total_with_smash += has_umag.sum()
    
    print(f"  Total candidates: {len(df)}")
    print(f"  With SMASH U magnitude: {has_umag.sum()}")
    print(f"  With SMASH U error: {has_uerr.sum()}")
    print(f"  Consistent magnitude/error pairs: {consistent_pairs.sum()}")
    print(f"  ❌ INCONSISTENT cases (error without magnitude): {inconsistent_count}")
    
    if inconsistent_count > 0:
        print("  WARNING: Still have inconsistent cases!")
        inconsistent_rows = df[~consistent_pairs]
        for idx, row in inconsistent_rows.head(3).iterrows():
            print(f"    Row {idx}: Umag={row['Usmashmag']}, Uerr={row['e_Usmashmag']}")
    else:
        print("  ✅ All magnitude/error pairs are consistent!")

print(f"\n{'='*50}")
print(f"SUMMARY:")
print(f"  Total candidates across both files: {total_candidates}")
print(f"  Total with SMASH U-band photometry: {total_with_smash}")
print(f"  Percentage with SMASH data: {100*total_with_smash/total_candidates:.1f}%")

print(f"\n✅ SUCCESS: SMASH DR2 U-band integration completed!")
print(f"Both candidate files now have consistent U magnitude and error data.")
print(f"Masked/invalid SMASH values have been properly handled as NaN values.")

FINAL VERIFICATION OF CANDIDATE FILES

Verifying: ./annas_candidates/final_lmc_ysgcands.csv
  Total candidates: 471
  With SMASH U magnitude: 433
  With SMASH U error: 433
  Consistent magnitude/error pairs: 471
  ❌ INCONSISTENT cases (error without magnitude): 0
  ✅ All magnitude/error pairs are consistent!

Verifying: ./annas_candidates/final_smc_ysgcands.csv
  Total candidates: 377
  With SMASH U magnitude: 377
  With SMASH U error: 377
  Consistent magnitude/error pairs: 377
  ❌ INCONSISTENT cases (error without magnitude): 0
  ✅ All magnitude/error pairs are consistent!

SUMMARY:
  Total candidates across both files: 848
  Total with SMASH U-band photometry: 810
  Percentage with SMASH data: 95.5%

✅ SUCCESS: SMASH DR2 U-band integration completed!
Both candidate files now have consistent U magnitude and error data.
Masked/invalid SMASH values have been properly handled as NaN values.
